# AoC 2024 Day 10 — Hoof It

**Spark — nine iterative joins up the height gradient**

Puzzle: <https://adventofcode.com/2024/day/10>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A topographic map: one digit per cell, height 0 (lowest) through 9 (highest).

A **hiking trail** starts at a 0, ends at a 9, moves only up/down/left/right — no diagonals — and gains exactly **one** height at every step. A **trailhead** is any height-0 cell, and its **score** is the number of distinct 9s reachable from it by some trail. Reaching the same 9 by several routes still scores 1.

- **Part 1** — sum the scores of every trailhead on the map.

## The approach

A graph search where **the depth is known in advance**, which is what makes it a clean fit for Spark.

Because every step must gain exactly one height, the trail graph is a DAG with exactly nine levels. There is no fixed point to iterate to and no visited-set to maintain — just `for height in range(1, 10)`, nine joins, done. That is the property worth noticing: most BFS-in-Spark writeups fight a `while` loop with a convergence check, and this puzzle hands you the loop bound for free.

The frontier carries **pairs**, not positions: `(trailhead, current cell)`. Carrying the origin through every join is what lets all 233 trailheads be searched in the same nine queries instead of one search per trailhead. At the end the frontier is exactly the relation "trailhead *t* can reach summit *s*", so the answer is `count()` — the sum of the scores never has to be computed as a sum at all.

Each step is `crossJoin` against a 4-row directions table (small enough that Spark broadcasts it) followed by an equi-join on the offset coordinates. The `.distinct()` after each step is doing real semantic work: a *score* counts **distinct reachable nines**, so collapsing duplicate paths as you go is both the correct definition and the thing that stops the frontier from growing path-count-shaped.

Honest scale note: nine sequential Spark stages over a 48×48 grid take about **1.2 s**, while the memoised DFS in `reference_python/y2024/day10.py` finishes in milliseconds. The relational shape is the one that would keep working on a grid that does not fit in memory; it is not the one that wins on this grid.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day10

spark = get_spark('aoc-2024-day10')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '89010123\n78121874\n87430965\n96549874\n45678903\n32019012\n01329801\n10456732\n'

print('part 1:', day10.part1(spark, EXAMPLE), '(expected 36)')

### Nine joins, counted

The cell below runs the same loop as `part1` but prints the frontier size after every height. Watch it widen as the trails fan out and then narrow as dead ends fail to join. The last table is each trailhead and its score — read it against the 5, 6, 5, 3, 1, 3, 5, 3, 5 in the puzzle.

In [ ]:
from pyspark.sql import functions as F

grid = day10.cells(spark, EXAMPLE).cache()
grid.groupBy('h').count().orderBy('h').show()

steps = spark.createDataFrame(day10.STEPS, 'dr INT, dc INT')
frontier = grid.filter(F.col("h") == 0).select(
    F.col("r").alias("sr"), F.col("c").alias("sc"), "r", "c"
)
print(f'height 0: {frontier.count():3d} (trailhead, position) pairs')

for height in range(1, 10):
    nxt = grid.filter(F.col("h") == height).select(
        F.col("r").alias("nr"), F.col("c").alias("nc")
    )
    frontier = (
        frontier.crossJoin(steps)
        .join(
            nxt,
            (F.col("nr") == F.col("r") + F.col("dr"))
            & (F.col("nc") == F.col("c") + F.col("dc")),
        )
        .select("sr", "sc", F.col("nr").alias("r"), F.col("nc").alias("c"))
        .distinct()
    )
    print(f'height {height}: {frontier.count():3d} pairs')

# The surviving frontier IS the answer relation: one row per
# (trailhead, reachable summit).
frontier.groupBy("sr", "sc").count().withColumnRenamed(
    "count", "score"
).orderBy("sr", "sc").show()
grid.unpersist()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 10)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day10.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day10 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **`try_cast`, not `cast`.** Spark 4 runs ANSI mode by default, where casting `"."` or `split`'s trailing `""` to `int` **raises** instead of returning null. `try_cast` nulls them so the `isNotNull` filter can drop them. This is the single most likely thing to break if you rewrite `cells()`.
- Those non-digit cells are removed from the relation entirely, which is exactly right: the smaller published examples use `.` as impassable terrain, and a cell that does not exist can never satisfy a join condition.
- The `.distinct()` belongs **inside** the loop, once per height. Move it to the end and you get a path count, not a score — a different number, and a different puzzle.
- `grid` is `cache()`d because the loop filters it ten times. Without it Spark re-splits and re-explodes the source string on every iteration.
- Nine chained joins build a deep lineage, and each `.distinct()` is a shuffle. On a real grid this is where you would consider checkpointing to cut the plan.
- Assumes single-digit heights 0–9 and strictly `+1` steps in the four cardinal directions. The step table is data, not code — adding diagonals means adding two rows to `STEPS`, nothing else.
- The final `count()` is the answer because a trailhead contributes one row per distinct summit. There is no separate summation step to get wrong.